### Skapar dataset med och utan de kategoriska features

In [1]:
import pandas as pd
df = pd.read_csv('revised_data.csv', index_col=0)
df = df.drop(columns='id') # Tar bort id eftersom id = index, onödig feature
df['gender'] = df['gender'].apply(lambda x: 'Woman' if x == 1 else 'Man') # Eftersom det är en kategorisk variabel gör jag den mer läsbar
yes_cat = df.drop(columns=[ 'ap_hi', 'ap_lo', 'height', 'weight', 'BMI'])
no_cat = df.drop(columns=['BMI_class', 'blood_pressure', 'height', 'weight'])

#One hot encoding
yes_cat = pd.get_dummies(yes_cat, columns=['BMI_class', 'blood_pressure', 'gender'])
no_cat = pd.get_dummies(no_cat, columns=['gender'])


### Använder custom class för att ta fram modeller att testa datan

In [2]:
from model_class import model_selection
#Initierar två klasser med och utan kategoriska features
X = yes_cat.drop(columns='cardio')
y = yes_cat['cardio']
with_cat = model_selection(X, y)
X = no_cat.drop(columns='cardio')
y = no_cat['cardio']
without_cat = model_selection(X, y)

In [3]:
#skalerar datan och gör en Train_test_split 
with_cat.standard_normal_scaling()
without_cat.standard_normal_scaling()


with_cat.train_test_split(test_size=0.33)
without_cat.train_test_split(test_size=0.33)


## Val av modeller
##### Eftersom detta är ett kvalificeringsproblem väljer jag följande:
- KNN
- Logistisk Regression
- Decision Trees
- Ridge Classifier
- RandomForestClassifier

In [4]:
#Skapar Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

#Ihopklistrad och omskriven kod. Ursprungligen från ChatGpt och sklearn
#Definiera Parametrar med respektive modell
param_grid = [
    {
        'classifier': [KNeighborsClassifier()],
        'classifier__n_neighbors': [1,3],
    },
    {
        'classifier': [LogisticRegression()],
        'classifier__C': [0.01, 0.1, 1]
    },
    {
        'classifier': [DecisionTreeClassifier()],
    },
    {
        'classifier': [RidgeClassifier()],
        'classifier__alpha': [0.01, 0.1, 1]
    },
    {
        'classifier': [RandomForestClassifier()],
        'classifier__max_depth': [1, 2],
    }
]
print('With Categories:')
with_cat.GridCV_pipeline(param_grid=param_grid)
print('Without Categories:')
without_cat.GridCV_pipeline(param_grid=param_grid)
#Tar ca 50 sekunder, RandomForest är långsam på cpu

With Categories:
Bästa modellen: Pipeline(steps=[('classifier', LogisticRegression(C=1))])
Bästa score: 0.6964857335895598
Without Categories:
Bästa modellen: Pipeline(steps=[('classifier', LogisticRegression(C=1))])
Bästa score: 0.7237873372463983
